# Contextual Retrieval：给孤立 Chunk 补上下文并绑定来源版本

**面试问题：为什么文档切块后关键词检索会失去语义，怎样做 Contextual Retrieval？**

## 回答主线

1. 文档 Chunk 常出现“该期限”“上述产品”等指代，脱离父标题后无法被查询词命中。
2. Contextual Retrieval 在索引前为每个 Chunk 生成短上下文，说明所属文档、章节、对象和时间。
3. 原始 Chunk 必须保持不变，上下文只作为检索字段，最终引用仍回到权威原文。
4. 手写 BM25 可以直接观察上下文新增词对每个文档分数的贡献。
5. 上下文、原文哈希和生成器版本必须绑定；源文档更新后旧上下文不能继续服务。
6. 评估应使用同一查询集合比较 Recall/MRR，并审计错误召回。

## 真实案例

六个制度 Chunk 分别来自耳机退款、企业发票和会员权益文档，正文都含“该期限/上述材料”等孤立指代。五个用户查询需要命中具体制度。我们从父标题和章节构造确定性上下文，手写 BM25 排名并复现源文档更新后上下文过期。这是可读的离线教学实验，用来验证协议和算法，不能外推为线上模型收益。

### 输入预览：六个孤立 Chunk 与父级元数据

In [1]:
import hashlib  # 导入哈希函数以绑定原文版本。
import math  # 导入对数函数以实现 BM25。
import re  # 导入正则表达式以做最小中英文分词。

chunks = [  # 构造六个具有父级语义的制度片段。
    {"id": "D1", "title": "耳机售后政策", "section": "退货期限", "text": "该期限自签收次日起计算，为7天。"},  # 正文没有再次出现耳机和退货。
    {"id": "D2", "title": "耳机售后政策", "section": "包装要求", "text": "上述商品需包装完整且配件齐全。"},  # 正文使用上述商品指代。
    {"id": "D3", "title": "企业发票指南", "section": "开票期限", "text": "该申请应在订单完成后30天内提交。"},  # 正文没有企业发票关键词。
    {"id": "D4", "title": "企业发票指南", "section": "所需材料", "text": "上述材料包括合同号和纳税人识别号。"},  # 正文使用上述材料指代。
    {"id": "D5", "title": "金卡会员权益", "section": "机场休息室", "text": "该权益每自然年可使用4次。"},  # 正文没有金卡和休息室关键词。
    {"id": "D6", "title": "金卡会员权益", "section": "积分有效期", "text": "该期限为获得积分后的24个月。"},  # 与其他期限片段产生歧义。
]  # 完成检索语料。
queries = [  # 定义五个真实查询及唯一相关 Chunk。
    {"q": "耳机退货几天", "relevant": "D1"},  # 查询售后期限。
    {"q": "耳机包装和配件要求", "relevant": "D2"},  # 查询包装条件。
    {"q": "企业发票多久申请", "relevant": "D3"},  # 查询开票期限。
    {"q": "企业发票需要合同号吗", "relevant": "D4"},  # 查询材料。
    {"q": "金卡机场休息室一年几次", "relevant": "D5"},  # 查询会员权益。
]  # 完成评估问题。
print("Chunk  父标题          章节       原文")  # 输出语料表头。
for chunk in chunks:  # 逐片段展示被切块后丢失的父级语义。
    print(f"{chunk['id']}     {chunk['title']:<12} {chunk['section']:<9} {chunk['text']}")  # 展示指代型正文。

Chunk  父标题          章节       原文
D1     耳机售后政策       退货期限      该期限自签收次日起计算，为7天。
D2     耳机售后政策       包装要求      上述商品需包装完整且配件齐全。
D3     企业发票指南       开票期限      该申请应在订单完成后30天内提交。
D4     企业发票指南       所需材料      上述材料包括合同号和纳税人识别号。
D5     金卡会员权益       机场休息室     该权益每自然年可使用4次。
D6     金卡会员权益       积分有效期     该期限为获得积分后的24个月。


## Baseline 基线：只索引原始 Chunk

In [2]:
def tokenize(text):  # 用字符二元组和连续字母数字做最小分词。
    normalized = re.sub(r"\s+", "", text.lower())  # 去除空白并统一英文大小写。
    chinese_bigrams = [normalized[index:index + 2] for index in range(max(0, len(normalized) - 1))]  # 生成相邻字符二元组。
    words = re.findall(r"[a-z0-9]+", normalized)  # 提取英文和数字词。
    return chinese_bigrams + words  # 返回用于教学 BM25 的词项。

def bm25_rank(query, documents, k1=1.5, b=0.75):  # 从词频和文档频率手写 BM25 排名。
    document_tokens = [tokenize(document) for document in documents]  # 分词全部索引文本。
    average_length = sum(len(tokens) for tokens in document_tokens) / len(document_tokens)  # 计算平均文档长度。
    query_terms = tokenize(query)  # 分词用户查询。
    rows = []  # 收集每篇文档的分项和总分。
    for document_index, tokens in enumerate(document_tokens):  # 逐文档计算分数。
        contributions = {}  # 保存命中词项的分数贡献。
        for term in set(query_terms):  # 查询重复词只计算一次。
            frequency = tokens.count(term)  # 计算当前文档词频。
            if frequency == 0:  # 未命中词项不产生贡献。
                continue  # 跳过当前词项。
            document_frequency = sum(term in other for other in document_tokens)  # 统计含该词项的文档数。
            inverse_frequency = math.log(1.0 + (len(documents) - document_frequency + 0.5) / (document_frequency + 0.5))  # 计算平滑 IDF。
            denominator = frequency + k1 * (1.0 - b + b * len(tokens) / average_length)  # 计算长度归一化分母。
            contributions[term] = inverse_frequency * frequency * (k1 + 1.0) / denominator  # 保存当前词项贡献。
        rows.append({"index": document_index, "score": sum(contributions.values()), "parts": contributions})  # 保存总分和分项。
    return sorted(rows, key=lambda row: (-row["score"], row["index"]))  # 按分数降序和原位置稳定排序。

raw_documents = [chunk["text"] for chunk in chunks]  # 只使用孤立原文建立基线索引。
baseline_top = []  # 收集五个查询的 Top-1。
for query in queries:  # 逐查询运行 BM25。
    ranking = bm25_rank(query["q"], raw_documents)  # 计算原文排名。
    baseline_top.append(chunks[ranking[0]["index"]]["id"])  # 保存 Top-1 Chunk ID。
print("查询                         目标  Baseline Top1")  # 输出基线结果表头。
for query, top in zip(queries, baseline_top):  # 逐查询展示失去父上下文的结果。
    print(f"{query['q']:<27} {query['relevant']}    {top}")  # 展示期限类 Chunk 互相混淆。

查询                         目标  Baseline Top1
耳机退货几天                      D1    D1
耳机包装和配件要求                   D2    D2
企业发票多久申请                    D3    D3
企业发票需要合同号吗                  D4    D4
金卡机场休息室一年几次                 D5    D1


### 核心实现：索引前上下文化与 BM25 分项

In [3]:
def contextualize(chunk, generator_version):  # 根据权威元数据生成短检索上下文。
    context = f"文档《{chunk['title']}》章节“{chunk['section']}”，内容对象是{chunk['title']}的{chunk['section']}。"  # 明确所属文档、章节和对象。
    source_hash = hashlib.sha256(chunk["text"].encode("utf-8")).hexdigest()[:12]  # 计算原文版本摘要。
    return {"context": context, "source_hash": source_hash, "generator_version": generator_version}  # 返回上下文和 provenance。

context_records = [contextualize(chunk, "context-template-r2") for chunk in chunks]  # 为六个 Chunk 生成版本化上下文。
contextual_documents = [record["context"] + chunk["text"] for chunk, record in zip(chunks, context_records)]  # 只在检索字段前置上下文并保留原文。
demo_ranking = bm25_rank(queries[0]["q"], contextual_documents)  # 对耳机退货问题观察 BM25 分项。
print("“耳机退货几天”上下文检索分项：")  # 输出最具解释性的查询。
for row in demo_ranking[:4]:  # 展示前四个候选的贡献词项。
    chunk = chunks[row["index"]]  # 读取当前候选元数据。
    print(f"chunk={chunk['id']} score={row['score']:.3f} parts={{{', '.join(f'{term}:{value:.2f}' for term, value in sorted(row['parts'].items()))}}}")  # 展示耳机、退货等上下文词贡献。
print("D1 context：", context_records[0])  # 展示原文哈希和生成器版本。

“耳机退货几天”上下文检索分项：
chunk=D1 score=3.668 parts={耳机:1.47, 退货:2.20}
chunk=D2 score=1.488 parts={耳机:1.49}
chunk=D3 score=0.000 parts={}
chunk=D4 score=0.000 parts={}
D1 context： {'context': '文档《耳机售后政策》章节“退货期限”，内容对象是耳机售后政策的退货期限。', 'source_hash': 'd175d2bfa572', 'generator_version': 'context-template-r2'}


## 结果解读：同一查询集合比较 Top-1 与 MRR

In [4]:
context_top = []  # 收集上下文索引的 Top-1。
reciprocal_ranks = []  # 收集每个相关 Chunk 的倒数排名。
for query in queries:  # 逐查询评估上下文索引。
    ranking = bm25_rank(query["q"], contextual_documents)  # 获取完整排名。
    ranked_ids = [chunks[row["index"]]["id"] for row in ranking]  # 转换为 Chunk ID 顺序。
    context_top.append(ranked_ids[0])  # 保存 Top-1。
    reciprocal_ranks.append(1.0 / (ranked_ids.index(query["relevant"]) + 1))  # 计算相关文档倒数排名。
baseline_accuracy = sum(top == query["relevant"] for top, query in zip(baseline_top, queries)) / len(queries)  # 计算原文 Top-1 准确率。
context_accuracy = sum(top == query["relevant"] for top, query in zip(context_top, queries)) / len(queries)  # 计算上下文 Top-1 准确率。
print("查询                         目标  RawTop1  ContextTop1")  # 输出同口径结果表头。
for query, raw_top, context_top_id in zip(queries, baseline_top, context_top):  # 逐查询对照两种索引。
    print(f"{query['q']:<27} {query['relevant']}    {raw_top:<7} {context_top_id}")  # 展示父语义恢复后的排名。
print(f"Top-1 {baseline_accuracy:.0%} -> {context_accuracy:.0%}，Context MRR={sum(reciprocal_ranks) / len(reciprocal_ranks):.3f}")  # 汇总检索质量。
print("解读：上下文增加的是检索线索，不替换证据；答案引用仍应指向原始 Chunk 和具体版本。")  # 明确 grounding 边界。

查询                         目标  RawTop1  ContextTop1
耳机退货几天                      D1    D1      D1
耳机包装和配件要求                   D2    D2      D2
企业发票多久申请                    D3    D3      D3
企业发票需要合同号吗                  D4    D4      D4
金卡机场休息室一年几次                 D5    D1      D5
Top-1 80% -> 100%，Context MRR=1.000
解读：上下文增加的是检索线索，不替换证据；答案引用仍应指向原始 Chunk 和具体版本。


## 失败案例：源文档更新后继续使用旧上下文版本

In [5]:
updated_chunk = chunks[0].copy()  # 复制 D1 以模拟政策更新。
updated_chunk["text"] = "该期限自签收次日起计算，为14天。"  # 将退款期限从七天更新为十四天。
old_record = context_records[0]  # 读取仍绑定旧原文的上下文记录。
new_hash = hashlib.sha256(updated_chunk["text"].encode("utf-8")).hexdigest()[:12]  # 计算新原文摘要。
stale = old_record["source_hash"] != new_hash  # 判断上下文和原文是否版本失配。
unsafe_answer = "7天"  # 模拟未失效旧索引仍返回历史原文。
rebuilt_record = contextualize(updated_chunk, "context-template-r2")  # 对新原文重新生成上下文记录。
safe_answer = "14天" if rebuilt_record["source_hash"] == new_hash else "拒答"  # 只有 provenance 匹配才使用更新证据。
print(f"旧hash={old_record['source_hash']} 新hash={new_hash} stale={stale} 盲用答案={unsafe_answer}")  # 展示静默陈旧问题。
print(f"重建记录={rebuilt_record} 安全答案={safe_answer}")  # 展示版本门禁后的正确政策。
print("修正策略：索引主键绑定 source_id + source_hash + contextualizer_version；任一变化都重建并原子切换快照。")  # 总结版本合同。

旧hash=d175d2bfa572 新hash=56e13bf93c8f stale=True 盲用答案=7天
重建记录={'context': '文档《耳机售后政策》章节“退货期限”，内容对象是耳机售后政策的退货期限。', 'source_hash': '56e13bf93c8f', 'generator_version': 'context-template-r2'} 安全答案=14天
修正策略：索引主键绑定 source_id + source_hash + contextualizer_version；任一变化都重建并原子切换快照。


### 生产边界与引用记录

In [6]:
citation = {"query": queries[0]["q"], "chunk_id": "D1", "source_hash": context_records[0]["source_hash"], "context_version": context_records[0]["generator_version"], "quote_source": "raw_chunk", "rank": 1}  # 构造可审计引用记录。
print("引用记录：", citation)  # 展示上下文索引和原文证据的分工。
print("生产替换点：真实系统还需 LLM 上下文生成、缓存、批处理、混合检索、重排、权限过滤、增量索引和离线标注集。")  # 明确模板上下文边界。

引用记录： {'query': '耳机退货几天', 'chunk_id': 'D1', 'source_hash': 'd175d2bfa572', 'context_version': 'context-template-r2', 'quote_source': 'raw_chunk', 'rank': 1}
生产替换点：真实系统还需 LLM 上下文生成、缓存、批处理、混合检索、重排、权限过滤、增量索引和离线标注集。


## 回归测试：最后只保护排名、原文与版本门禁

In [7]:
assert context_accuracy == 1.0 and context_accuracy > baseline_accuracy  # 验证五个查询的上下文 Top-1 全部正确且优于原文基线。
assert all(chunk["text"] in document for chunk, document in zip(chunks, contextual_documents))  # 验证上下文化没有替换或丢失原始证据。
assert context_records[0]["source_hash"] == hashlib.sha256(chunks[0]["text"].encode("utf-8")).hexdigest()[:12]  # 验证上下文绑定原文哈希。
assert stale and unsafe_answer != safe_answer and safe_answer == "14天"  # 验证陈旧索引反例与重建修正。
assert citation["quote_source"] == "raw_chunk" and citation["rank"] == 1  # 验证最终引用回到权威原文。
print("回归测试通过：上下文排名、原文保留、哈希绑定、陈旧失效和引用 provenance 均成立。")  # 用少量断言总结检索合同。

回归测试通过：上下文排名、原文保留、哈希绑定、陈旧失效和引用 provenance 均成立。
